In [25]:
import torch
from torch.utils.data import DataLoader,TensorDataset
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_breast_cancer

In [3]:
x,y = load_breast_cancer(return_X_y=True)
X_train,X_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=7,stratify=y)
scaler = StandardScaler()
scaler.fit(X_train)
X_train_sca = scaler.transform(X_train)
X_test_sca = scaler.transform(X_test)

In [5]:
X_train

array([[1.258e+01, 1.840e+01, 7.983e+01, ..., 8.772e-03, 2.505e-01,
        6.431e-02],
       [2.055e+01, 2.086e+01, 1.378e+02, ..., 2.148e-01, 3.077e-01,
        7.569e-02],
       [2.463e+01, 2.160e+01, 1.655e+02, ..., 2.475e-01, 3.157e-01,
        9.671e-02],
       ...,
       [1.176e+01, 1.814e+01, 7.500e+01, ..., 7.160e-02, 1.978e-01,
        6.915e-02],
       [1.026e+01, 1.222e+01, 6.575e+01, ..., 6.696e-02, 2.937e-01,
        7.722e-02],
       [1.385e+01, 1.721e+01, 8.844e+01, ..., 5.104e-02, 2.364e-01,
        7.182e-02]], shape=(455, 30))

In [6]:
y_train

array([1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1,
       0, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0,
       1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1,
       0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1,
       1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1,
       0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1,
       0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0,
       0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1,
       1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1,
       0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0,
       0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 1, 0, 0,
       0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1,
       0, 1, 1, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0,
       0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1,

In [8]:
X_test_sca_tensor = torch.from_numpy(X_test_sca).float()
X_train_sca_tensor = torch.from_numpy(X_train_sca).float()
y_train_tensor = torch.from_numpy(y_train).float().unsqueeze(1)
y_test_tensor = torch.from_numpy(y_test).float().unsqueeze(1)

In [13]:
bc = load_breast_cancer()

print(bc.feature_names)
print(bc.target_names)

['mean radius' 'mean texture' 'mean perimeter' 'mean area'
 'mean smoothness' 'mean compactness' 'mean concavity'
 'mean concave points' 'mean symmetry' 'mean fractal dimension'
 'radius error' 'texture error' 'perimeter error' 'area error'
 'smoothness error' 'compactness error' 'concavity error'
 'concave points error' 'symmetry error' 'fractal dimension error'
 'worst radius' 'worst texture' 'worst perimeter' 'worst area'
 'worst smoothness' 'worst compactness' 'worst concavity'
 'worst concave points' 'worst symmetry' 'worst fractal dimension']
['malignant' 'benign']


In [15]:
train_ds = TensorDataset(X_train_sca_tensor,y_train_tensor)

In [17]:
X_train_sca_tensor.shape

torch.Size([455, 30])

In [19]:
loader = DataLoader(dataset=train_ds,batch_size=32,shuffle=True)

In [21]:
class BCNet(torch.nn.Module):
    def __init__(self):
        super(BCNet,self).__init__()
        self.fc1 = nn.Linear(30,128)
        self.fc2 = nn.Linear(128,32)
        self.fc3 = nn.Linear(32,1)

    def forward(self,x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.sigmoid(self.fc3(x))
        return x

In [23]:
model = BCNet()
creteria = nn.BCELoss()
optimizer = optim.Adam(model.parameters(),lr=0.001)

In [28]:
epochs = 20
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for batch_X, batch_y in loader:
        optimizer.zero_grad()

        preds = model(batch_X)
        loss = creteria(preds,batch_y) #it use the magic function __call__

        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f'Epoch {epoch+1}: Loss: {running_loss/len(loader):.3f}')

Epoch 1: Loss: 0.580
Epoch 2: Loss: 0.354
Epoch 3: Loss: 0.183
Epoch 4: Loss: 0.123
Epoch 5: Loss: 0.118
Epoch 6: Loss: 0.075
Epoch 7: Loss: 0.098
Epoch 8: Loss: 0.062
Epoch 9: Loss: 0.057
Epoch 10: Loss: 0.053
Epoch 11: Loss: 0.050
Epoch 12: Loss: 0.056
Epoch 13: Loss: 0.045
Epoch 14: Loss: 0.061
Epoch 15: Loss: 0.041
Epoch 16: Loss: 0.043
Epoch 17: Loss: 0.037
Epoch 18: Loss: 0.035
Epoch 19: Loss: 0.033
Epoch 20: Loss: 0.031


In [27]:
print("Training X:", X_train_sca_tensor.shape)
print("Training y:", y_train_tensor.shape)

print("Dataset X:", train_ds.tensors[0].shape)
print("Dataset y:", train_ds.tensors[1].shape)

batch_X, batch_y = next(iter(loader))

print("Batch X:", batch_X.shape)
print("Batch y:", batch_y.shape)

Training X: torch.Size([455, 30])
Training y: torch.Size([455, 1])
Dataset X: torch.Size([455, 30])
Dataset y: torch.Size([455, 1])
Batch X: torch.Size([32, 30])
Batch y: torch.Size([32, 1])


In [35]:
model.eval()
with torch.no_grad():
    preds = model(X_test_sca_tensor)
    loss = creteria(preds, y_test_tensor).item()

    predicted_labels = (preds >= 0.5).float()
    accuracy = (predicted_labels == y_test_tensor).float().mean().item()

print(f"Test loss: {loss:.4f} | Test accuracy: {accuracy:.4f}")

Test loss: 0.0352 | Test accuracy: 0.9825


In [36]:
accuracy

0.9824561476707458